In [2]:
import transformers
from transformers import AutoTokenizer, AutoModel
import torch
import pandas as pd
import pickle
from tqdm import tqdm

In [12]:
device = "cuda"

In [4]:
df = pd.read_csv("data/prot_id_content.csv")[:100]



In [5]:
len(df["content"])

100

In [8]:
tokenizer = AutoTokenizer.from_pretrained("facebook/esm2_t12_35M_UR50D", do_lower_case=False)
model = AutoModel.from_pretrained("facebook/esm2_t12_35M_UR50D").to(device)
model.eval()
emb_dict = {}
for i in tqdm(range(len(df["content"]))):
    encoded_input = tokenizer(df["content"][i], return_tensors="pt")
    encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
    with torch.no_grad():
        output = model(**encoded_input)

    protein_embedding = output.last_hidden_state
    protein_embedding = protein_embedding.squeeze()
    protein_embedding = protein_embedding.mean(dim=0)
    emb_dict[df['id'][i].item()] = protein_embedding.half().cpu()       # x: [D] или [1,D]


Loading weights: 100%|██████████| 209/209 [00:00<00:00, 1383.22it/s, Materializing param=encoder.layer.11.output.dense.weight]                      
EsmModel LOAD REPORT from: facebook/esm2_t12_35M_UR50D
Key                         | Status     | 
----------------------------+------------+-
esm.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
pooler.dense.bias           | MISSING    | 
pooler.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
100%|██████████| 100/100 [00:04<00:00, 20.60it/s]


In [7]:
with open("data/dict.pkl", "wb") as f:
    pickle.dump(emb_dict, f)

In [11]:
# Load model directly


tokenizer = AutoTokenizer.from_pretrained("zhihan1996/DNABERT-S", trust_remote_code=True)
model = AutoModel.from_pretrained("zhihan1996/DNABERT-S", trust_remote_code=True).to(device)

dna = "CUGCUGCUGCUGCUGCUG"
inputs = tokenizer(dna, return_tensors = 'pt')["input_ids"]
output = model(**encoded_input)

# Эмбеддинги (последний слой)
dna_embeddings = output.last_hidden_state
dna_embeddings = dna_embeddings.squeeze()
dna_embeddings

RuntimeError: Tensor on device meta is not on the expected device cpu!

In [14]:
tokenizer = AutoTokenizer.from_pretrained("DeepChem/ChemBERTa-100M-MLM")
model = AutoModel.from_pretrained("DeepChem/ChemBERTa-100M-MLM").to(device)
model.eval()
buf_sm = []
for i in df["content"]:
    encoded_input = tokenizer(i, return_tensors="pt")
    encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
    with torch.no_grad():
        output = model(**encoded_input)

    sm_embedding = output.last_hidden_state
    sm_embedding = sm_embedding.squeeze()
    sm_embedding = sm_embedding.mean(dim=0)
    buf_sm.append(sm_embedding.half().cpu())        # x: [D] или [1,D]

T_sm = torch.stack(buf_sm, dim=0)  # итоговый [N,D]

Loading weights: 100%|██████████| 197/197 [00:00<00:00, 1359.77it/s, Materializing param=encoder.layer.11.output.dense.weight]              
RobertaModel LOAD REPORT from: DeepChem/ChemBERTa-100M-MLM
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
pooler.dense.bias         | MISSING    | 
pooler.dense.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Kernel Name: _ZN2at6native34_scatter_gather_elementwise_kernelILi256ELi4EZNS0_36_cuda_scatter_gather_internal_kernelILb0ENS0_10OpaqueTypeILi8EEElEclINS0_12TensorAssignEEEvRNS_14TensorIteratorElllRKT_EUliE_EEviT1_
VGPU=0x2f6227e0 SWq=0x76ad066b3000, HWq=0x76aa81100000, id=1
	Dispatch Header =0xb02 (type=2, barrier=1, acquire=1, release=1), setup=0
	grid=[256, 1, 1], workgroup=[256, 1, 1]
	private_seg_size=0, group_seg_size=0
	kernel_obj=0x76aa46fb3600, kernarg_address=0x0x76aa45c12980
	completion_signal=0x0, correlation_id=0
	rptr=75581, wptr=75592
 

:0:rocdevice.cpp            :3580: 31380488065 us:  Callback: Queue 0x76aa81100000 aborting with error : HSA_STATUS_ERROR_EXCEPTION: An HSAIL operation resulted in a hardware exception. code: 0x1016
/home/askakolbaska/graph_LP/.venv/lib/python3.13/site-packages/transformers/masking_utils.py:325: UserWarning: HIP warning: unspecified launch failure (Triggered internally at /pytorch/aten/src/ATen/hip/impl/HIPGuardImplMasqueradingAsCUDA.h:83.)
  and (padding_mask is None or padding_mask.all())


AcceleratorError: HIP error: unspecified launch failure
Search for `hipErrorLaunchFailure' in https://rocm.docs.amd.com/projects/HIP/en/latest/index.html for more information.
HIP kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing AMD_SERIALIZE_KERNEL=3
Compile with `TORCH_USE_HIP_DSA` to enable device-side assertions.


In [11]:
tokenizer = AutoTokenizer.from_pretrained("PharMolix/BioMedGPT-LM-7B")
model = AutoModel.from_pretrained("PharMolix/BioMedGPT-LM-7B").to(device)
model.eval()
emb_dict = {}
for i in tqdm(range(len(df["content"]))):
    encoded_input = tokenizer(df["content"][i], return_tensors="pt")
    encoded_input = {k: v.to(device) for k, v in encoded_input.items()}
    with torch.no_grad():
        output = model(**encoded_input)

    protein_embedding = output.last_hidden_state
    protein_embedding = protein_embedding.squeeze()
    protein_embedding = protein_embedding.mean(dim=0)
    emb_dict[df['id'][i].item()] = protein_embedding.half().cpu()

Loading weights: 100%|██████████| 290/290 [00:00<00:00, 1543.59it/s, Materializing param=norm.weight]                              
LlamaModel LOAD REPORT from: PharMolix/BioMedGPT-LM-7B
Key            | Status     |  | 
---------------+------------+--+-
lm_head.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
  2%|▏         | 2/100 [01:20<1:05:53, 40.34s/it]


KeyboardInterrupt: 